# Notebook 39 - Development-slice graph, prevention endpoints, allocation control

Three arms answering the third review. Readings fixed in stage 2.

**A, data independence for V-C on the second corpus.** The vulnerability graph is rebuilt from a 10% development slice of the TRAINING rows and V-C is recomputed with it; its validity against notebook 33's validation harms is compared with the validation-graph V-C. A1: within 0.05 on HSR harm on both architectures.

**B, the prevention claims with the full deployment audit.** The six collapsing CICIoT2023 cells are recovered unclipped, with clipped inputs, and (dense cells) with GroupNorm, recording benign escalation, attack-miss rate, family and fine macro-F1, ground-truth HSR and AWBIR at every unit and after recalibration of the final checkpoint. B1: clipping's final HSR within 0.02 of the recalibrated unclipped student's in five of six cells. B2: GroupNorm attack-miss below 0.10 and family F1 above 0.50 at every unit.

**C, allocation versus channel identity at depth.** For each frozen deep structure, a random structure with the same per-layer widths is recovered over twenty seeds and recalibrated. C1: allocation-matched random reproduces the method's mean HSR within 0.006 for four of five allocations.

**Stages.** 1 bootstrap, 2 pre-registration, 3 arm A (IoMT), 4 CICIoT2023 helpers, 5 arm B (resumable), 6 arm C (resumable), 7 verdicts, 8 figure. GPU; about two and a half hours.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, hashlib, itertools, gc
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.saber.bridge_ciciot import load_bridge
from src.saber.bridge_iomt import build_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import enumerate_cnn1d_channel_groups, prune_cnn1d_channels, count_parameters, profile_forward_flops
from src.saber.leverage import semantic_boundary_leverage
from src.saber.risk_graph import build_alert_semantic_vulnerability_graph, aggregate_robust_edge_weights

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; OUT = R / "39_endpoint_prevention_allocation"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"; IOMT_MODEL_DIR = REPO / "models/iomt"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "A_dev_graph__B_prevention_endpoints__C_allocation",
    "A_dev_graph": {
        "design": ("CIC-IoMT-2024: a development slice of 10% of the TRAINING rows (stratified by class, seed 39) is set aside; the "
                   "vulnerability graph is rebuilt from the shallow teacher's logits on that slice with the released configuration "
                   "(top-3, T=1, weights 0.5/0.5, support 20, CVaR 0.75); semantic boundary leverage for both architectures is "
                   "recomputed with the development graph on a class-balanced sample of the REMAINING training rows (512 per class, "
                   "seed 7); V-C_dev = within-layer percentile rank of that leverage times the layer-mean Fisher of notebook 33 "
                   "(training-based, unchanged); correlated with notebook 33's single-channel harms on the validation subsample"),
        "A1": ("V-C_dev's Spearman correlation with single-channel HSR harm is within 0.05 of the validation-graph V-C's on each "
               "architecture (0.509 shallow, 0.510 deep); if so, V-C's out-of-sample validity on the ground-truth endpoint does not "
               "depend on the validation-built graph and the 'out of sample' claim is restored for V-C on the graph-free harms")},
    "B_prevention_endpoints": {
        "design": ("CICIoT2023, the six collapsing cells of notebook 35 (shallow fisher/V-C/dense at seed 307; deep fisher/V-C/dense at "
                   "seed 401), recovered exactly as there (10% seeded subset, 8 or 6 units, Adam 1e-3, class-weighted CE) under three "
                   "conditions, unclipped baseline, inputs clipped at the 0.1/99.9 training percentiles (400k-row sample, seed 2026), "
                   "and GroupNorm dense models for the two dense cells; the FULL deployment audit (benign escalation, attack-miss rate, "
                   "family and fine macro-F1, ground-truth HSR, AWBIR) recorded at every unit, plus the recalibrated audit of every final "
                   "checkpoint"),
        "B1": ("at the final unit, the clipped student's ground-truth HSR is within 0.02 of the unclipped-then-recalibrated student's HSR "
               "in at least five of six cells; if not, clipping has a measurable deployment cost and the paper reports it"),
        "B2": ("GroupNorm dense models keep attack-miss rate below 0.10 and family macro-F1 above 0.50 at every unit; if not, the "
               "'no escalation spike' reading is qualified by what they do instead")},
    "C_allocation": {
        "design": ("CICIoT2023 deep, the five frozen 20b structures; for each, an ALLOCATION-MATCHED RANDOM structure keeps the same "
                   "number of channels in every layer but chooses which channels to keep uniformly at random (seed 39 + method index); "
                   "5 allocations x 20 seeds under minimal recovery, every model recalibrated on the notebook 31 slice; compared with "
                   "notebook 37's twenty-seed means for the same allocations' methods"),
        "C1": ("for at least four of the five allocations, the allocation-matched random mean recalibrated HSR is within 0.006 (twice "
               "the twenty-seed half-width) of the method's own mean; if so, the deployed ground-truth differences among methods at "
               "depth are attributable to layer-width allocation rather than channel identity, and the paper says so; if not, channel "
               "identity contributes and the paper says that")},
    "seeds": [101, 211, 307, 401, 503, 613, 719, 823, 907, 1013, 1109, 1201, 1303, 1409, 1511, 1607, 1709, 1801, 1907, 2003],
    "no_test_access": True,
}
(OUT / "P39_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2)); print(json.dumps(PREREG, indent=2))
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]; SEEDS20 = PREREG["seeds"]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; MIN_W = 8; COLLAPSE_B2A = 0.10


In [ ]:
# Stage 3 - A: IoMT development-slice graph and V-C validity on graph-free harms
I_TRAIN, I_VAL, I_CLASSES, i_tax, I_MAN = build_bridge(REPO / "data/iomt_bridge"); I_N = len(I_CLASSES)


def make_iomt(arch):
    if arch == "shallow":
        class CNN1D(nn.Module):
            def __init__(self, n):
                super().__init__()
                self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64), nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
                self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n)
            def forward(self, x):
                if x.dim() == 2:
                    x = x.unsqueeze(1)
                return self.head(self.pool(self.conv(x.float())).squeeze(-1))
        return CNN1D(I_N)
    class DeepCNN1D(nn.Module):
        def __init__(self, n):
            super().__init__()
            def blk(i, o):
                return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
            self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    return DeepCNN1D(I_N)


I_TEACHERS = {}
for arch in ["shallow", "deep"]:
    m = make_iomt(arch); m.load_state_dict(torch.load(IOMT_MODEL_DIR / f"{arch}_teacher_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
    I_TEACHERS[arch] = m.to(DEVICE).eval()
iXt, iYt = I_TRAIN.dataset.tensors; iyt = iYt.numpy(); N_IT = len(iXt)
# development slice: 10% of training rows, stratified, seed 39
rng = np.random.default_rng(39); dev_mask = np.zeros(N_IT, dtype=bool)
for c in range(I_N):
    idx = np.where(iyt == c)[0]; rng.shuffle(idx); dev_mask[idx[: int(round(0.10 * len(idx)))]] = True
DEV_X, DEV_Y = iXt[torch.from_numpy(dev_mask)], iyt[dev_mask]; print("development rows:", int(dev_mask.sum()), "| remaining training rows:", int((~dev_mask).sum()))
with torch.no_grad():
    DEV_LOGITS = torch.cat([I_TEACHERS["shallow"](DEV_X[i:i + 8192].to(DEVICE)).cpu() for i in range(0, len(DEV_X), 8192)]).numpy()
frames = [build_alert_semantic_vulnerability_graph(DEV_LOGITS, DEV_Y, i_tax, prof, top_k=3, temperature=1.0, confusion_weight=0.5, margin_weight=0.5, min_class_support=20)
          for prof in DEFAULT_COST_PROFILES.values()]
graph_dev = aggregate_robust_edge_weights(pd.concat(frames, ignore_index=True), method="cvar", q=0.75)
assert "robust_weight" in graph_dev.columns, f"unexpected aggregate columns: {list(graph_dev.columns)}"
graph_dev.to_csv(OUT / "A_asvg_edges_robust_dev.csv", index=False)
graph_val = pd.read_csv(R / "32_iomt_bridge/asvg_edges_robust.csv")
print("dev graph edges:", len(graph_dev), "| validation graph edges:", len(graph_val))

# balanced SBL calibration sample from the remaining training rows (512 per class, seed 7)
rem = np.where(~dev_mask)[0]; r7 = np.random.default_rng(7)
bal = np.concatenate([r7.permutation(rem[iyt[rem] == c])[:512] for c in range(I_N) if (iyt[rem] == c).sum() > 0])
CAL_BAL = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(iXt[bal], iYt[bal]), batch_size=1024, shuffle=False)
I_EXAMPLE = I_VAL.dataset.tensors[0][:8].float().to(DEVICE)
rows = []
for arch, model in I_TEACHERS.items():
    groups = enumerate_cnn1d_channel_groups(model, I_EXAMPLE)
    sbl_t, _ = semantic_boundary_leverage(model, CAL_BAL, groups, graph_dev, device=DEVICE, max_samples_per_class=512, edge_weight_column="robust_weight", normalize_by_group_size=0.0, normalize_by_flops=0.0)
    col = [c for c in sbl_t.columns if c != "group_id" and np.issubdtype(sbl_t[c].dtype, np.number)][0]
    s33 = pd.read_csv(R / f"33_iomt_scores_structures/{arch}_scores.csv"); s33["group_id"] = s33["group_id"].astype(str); sbl_t["group_id"] = sbl_t["group_id"].astype(str)
    s = s33.merge(sbl_t[["group_id", col]].rename(columns={col: "sbl_dev"}), on="group_id", validate="1:1")
    s["vc_dev"] = s.groupby("module_path")["sbl_dev"].rank(pct=True) * s["module_path"].map(s.groupby("module_path")["fisher"].mean())
    harms = pd.read_csv(R / f"33_iomt_scores_structures/{arch}_single_group_causal_ablation.csv"); harms["group_id"] = harms["group_id"].astype(str)
    d = s.merge(harms.drop(columns=[k for k in ("module_path", "channel_index") if k in harms.columns]), on="group_id", validate="1:1"); d.to_csv(OUT / f"A_{arch}_scores_dev.csv", index=False)
    for harm in ["harm_hsr_balanced_soc", "harm_family_macro_f1", "harm_fine_macro_f1", "harm_awbir"]:
        rows.append({"architecture": arch, "harm": harm, "vc_val_graph": float(spearmanr(d["saber_v2"], d[harm]).statistic), "vc_dev_graph": float(spearmanr(d["vc_dev"], d[harm]).statistic),
                     "fisher": float(spearmanr(d["fisher"], d[harm]).statistic), "taylor": float(spearmanr(d["taylor"], d[harm]).statistic), "magnitude": float(spearmanr(d["magnitude"], d[harm]).statistic)})
    rows.append({"architecture": arch, "harm": "score_agreement_vc_dev_vs_vc_val", "vc_val_graph": float("nan"), "vc_dev_graph": float(spearmanr(d["vc_dev"], d["saber_v2"]).statistic), "fisher": float("nan"), "taylor": float("nan"), "magnitude": float("nan")})
A = pd.DataFrame(rows); A.to_csv(OUT / "A_validity_dev_graph.csv", index=False); print(A.round(3).to_string(index=False))
hsr = A[A.harm == "harm_hsr_balanced_soc"].set_index("architecture")
A1 = bool(all(abs(hsr.loc[a, "vc_dev_graph"] - hsr.loc[a, "vc_val_graph"]) <= 0.05 for a in ["shallow", "deep"]))
print("A1 (V-C validity on HSR harm within 0.05 of the validation-graph value on both architectures):", A1)
del I_TRAIN, I_VAL, iXt, iYt, DEV_X; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None


In [ ]:
# Stage 4 - CICIoT2023 data, teachers and helpers (shared by B and C)
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES); SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_WA = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}


def make_cnn(norm, n_classes=34):
    def N(c):
        return nn.BatchNorm1d(c) if norm == "bn" else nn.GroupNorm(1, c)
    class CNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), N(64), nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), N(128))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    class DeepCNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            def blk(i, o):
                return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), N(o)]
            self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    return {"shallow": CNN1D, "deep": DeepCNN1D}


TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = make_cnn("bn")["deep"](); _dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()
Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy(); Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)
_sub = Xt[torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(2026))[:400_000]].numpy()
LO = torch.tensor(np.percentile(_sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE); HI = torch.tensor(np.percentile(_sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
del _sub


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


def forward_logits(model, X=None):
    X = EX_X if X is None else X; model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone()) for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}


def audit_from_logits(lg, arch):
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES); aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]), "fine_f1": float(a["fine_macro_f1"]),
            "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]), "hsr_fatigue": float(a["hsr_alert_fatigue"]), "awbir": float(aw)}


def audit(model, arch, mode="eval"):
    return audit_from_logits(forward_logits(model) if mode == "eval" else forward_logits_batch_stats(model), arch)


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def removed_path(arch, method):
    return (R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv") if arch == "shallow" else (R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv")


def raw_student(arch, method):
    rm = pd.read_csv(removed_path(arch, method)); pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_WA[arch]); return st.to(DEVICE)


def make_loader(subset_seed, order_seed=None, batch=1024):
    order_seed = subset_seed if order_seed is None else order_seed
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(subset_seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=batch, shuffle=True, generator=torch.Generator().manual_seed(order_seed))


def recover(model, arch, loader, n_units, weights=None, tag="", has_bn=True):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=weights) if weights is not None else nn.CrossEntropyLoss(); rows = []
    for unit in range(1, n_units + 1):
        for xb, yb in loader:
            model.train(); opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        ev = audit(model, arch, "eval"); bs = audit(model, arch, "batch") if has_bn else None
        col = bool(ev["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A) if has_bn else bool(ev["b2a"] > COLLAPSE_B2A)
        rows.append({"tag": tag, "architecture": arch, "unit": unit, **{f"eval_{k}": v for k, v in ev.items()}, "batch_b2a": (bs["b2a"] if has_bn else float("nan")), "is_collapse": col})
    return rows


E_MAX = {"shallow": 8, "deep": 6}
PRUNED_CELLS = [("shallow", "fisher", 307), ("shallow", "saber_v2", 307), ("deep", "fisher", 401), ("deep", "saber_v2", 401)]; DENSE_CELLS = [("shallow", 307), ("deep", 401)]
JOBS = [(f"pruned/{a}/{m}/s{s}", a, s, ("pruned", m)) for a, m, s in PRUNED_CELLS] + [(f"dense/{a}/s{s}", a, s, ("dense", None)) for a, s in DENSE_CELLS]
print("CICIoT2023 ready | eval rows", len(_idx), "| clip bounds computed on 400k training rows")


In [ ]:
# Stage 5 - B: unclipped baseline, clipped inputs, GroupNorm; full audit at every unit and after recalibration of the final checkpoint (resumable)
EP = OUT / "B_epochs.csv"; FINAL = OUT / "B_final.csv"
ep_rows = pd.read_csv(EP).to_dict("records") if EP.exists() else []; fin_rows = pd.read_csv(FINAL).to_dict("records") if FINAL.exists() else []
done = {(r["condition"], r["tag"]) for r in fin_rows}
for condition in ["baseline", "clipped"]:
    for tag, arch, seed, (kind, method) in JOBS:
        if (condition, tag) in done: continue
        torch.manual_seed(seed); np.random.seed(seed)
        inner = raw_student(arch, method) if kind == "pruned" else copy.deepcopy(TEACHERS[arch]).to(DEVICE)
        model = Clipped(inner, LO, HI).to(DEVICE) if condition == "clipped" else inner
        ep = recover(model, arch, make_loader(seed), E_MAX[arch], weights=CLASS_W, tag=tag)
        for r in ep: r["condition"] = condition
        final_trained = audit(model, arch, "eval"); recalibrate_bn(model); final_recal = audit(model, arch, "eval")
        ep_rows.extend(ep); fin_rows.append({"condition": condition, "tag": tag, "architecture": arch, "collapses": int(sum(r["is_collapse"] for r in ep)),
                                              **{f"trained_{k}": v for k, v in final_trained.items()}, **{f"recal_{k}": v for k, v in final_recal.items()}})
        pd.DataFrame(ep_rows).to_csv(EP, index=False); pd.DataFrame(fin_rows).to_csv(FINAL, index=False)
        print(f"{condition:8s} {tag:30s}: collapses {fin_rows[-1]['collapses']} | final a2b {final_trained['a2b']:.3f} HSR {final_trained['hsr_balanced']:.3f} | recal a2b {final_recal['a2b']:.3f} HSR {final_recal['hsr_balanced']:.3f}")
for arch, seed in DENSE_CELLS:
    tag = f"groupnorm_dense/{arch}/s{seed}"
    if ("groupnorm", tag) in done: continue
    ck = MODEL_DIR / f"{arch}_groupnorm_teacher_seed0.pt"; m = make_cnn("gn")[arch]()
    if ck.exists():
        m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); m = m.to(DEVICE).eval()
    else:
        torch.manual_seed(0); np.random.seed(0); m = m.to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
        for _ in range(4):
            m.train()
            for xb, yb in TRAIN_LOADER:
                opt.zero_grad(); lossf(m(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        m.eval(); torch.save({"state_dict": m.cpu().state_dict()}, ck); m = m.to(DEVICE).eval()
    torch.manual_seed(seed); np.random.seed(seed); model = copy.deepcopy(m).to(DEVICE)
    ep = recover(model, arch, make_loader(seed), E_MAX[arch], weights=CLASS_W, tag=tag, has_bn=False)
    for r in ep: r["condition"] = "groupnorm"
    final_trained = audit(model, arch, "eval"); ep_rows.extend(ep)
    fin_rows.append({"condition": "groupnorm", "tag": tag, "architecture": arch, "collapses": int(sum(r["is_collapse"] for r in ep)), **{f"trained_{k}": v for k, v in final_trained.items()}, **{f"recal_{k}": float("nan") for k in final_trained}})
    pd.DataFrame(ep_rows).to_csv(EP, index=False); pd.DataFrame(fin_rows).to_csv(FINAL, index=False)
    print(f"groupnorm {tag:30s}: spikes {fin_rows[-1]['collapses']} | final a2b {final_trained['a2b']:.3f} famF1 {final_trained['family_f1']:.3f} HSR {final_trained['hsr_balanced']:.3f}")
print("B rows:", len(fin_rows))


In [ ]:
# Stage 6 - C: allocation-matched random structures at depth, 20 seeds each (resumable)
DENSE_W = {"conv.0": 64, "conv.3": 128, "conv.7": 128, "conv.10": 256}
ALLOC = OUT / "C_allocations.csv"; alloc_rows = []
for mi, method in enumerate(METHODS):
    rm = pd.read_csv(removed_path("deep", method)); removed_per_layer = rm.groupby("module_path").size().to_dict()
    r = np.random.default_rng(39 + mi); pm = {}
    for layer, n_dense in DENSE_W.items():
        k = int(removed_per_layer.get(layer, 0))
        if k > 0: pm[layer] = sorted(r.choice(n_dense, k, replace=False).tolist())
    st, _ = prune_cnn1d_channels(TEACHERS["deep"], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_WA["deep"]); st = st.to(DEVICE)
    pd.DataFrame([{"module_path": l, "channel_index": c} for l, cs in pm.items() for c in cs]).to_csv(OUT / f"C_deep_{method}_allocation_random_removed_groups.csv", index=False)
    widths = "/".join(str(DENSE_W[l] - int(removed_per_layer.get(l, 0))) for l in DENSE_W)
    m0 = profile_forward_flops(TEACHERS["deep"], EXAMPLE_INPUT)["flops_per_item"]; raw = audit(st, "deep", "eval")
    alloc_rows.append({"allocation_of": method, "widths": widths, "parameters": int(count_parameters(st)), "flop_reduction": float(1 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / m0), "raw_hsr_balanced": raw["hsr_balanced"], "raw_awbir": raw["awbir"]})
    print(f"allocation of {method:9s}: widths {widths} params {alloc_rows[-1]['parameters']} flop-red {alloc_rows[-1]['flop_reduction']:.3f}")
pd.DataFrame(alloc_rows).to_csv(ALLOC, index=False)

RUNS = OUT / "C_runs.csv"; rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []; done = {(r["allocation_of"], r["seed"]) for r in rows}
for method in METHODS:
    rm = pd.read_csv(OUT / f"C_deep_{method}_allocation_random_removed_groups.csv"); pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    for seed in SEEDS20:
        if (method, seed) in done: continue
        torch.manual_seed(seed); np.random.seed(seed)
        st, _ = prune_cnn1d_channels(TEACHERS["deep"], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_WA["deep"]); st = st.to(DEVICE)
        opt = torch.optim.Adam(st.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W); st.train()
        for xb, yb in make_loader(seed):
            opt.zero_grad(); lossf(st(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        tr = audit(st, "deep", "eval"); recalibrate_bn(st); rc = audit(st, "deep", "eval")
        rows.append({"allocation_of": method, "seed": seed, **{f"trained_{k}": v for k, v in tr.items()}, **{f"recal_{k}": v for k, v in rc.items()}})
        pd.DataFrame(rows).to_csv(RUNS, index=False); print(f"allocation of {method:9s} s{seed:4d}: HSR recal={rc['hsr_balanced']:.4f} awbir recal={rc['awbir']:.4f}")
print("C runs:", len(rows))


In [ ]:
# Stage 7 - verdicts
A = pd.read_csv(OUT / "A_validity_dev_graph.csv"); hsr = A[A.harm == "harm_hsr_balanced_soc"].set_index("architecture")
A1 = bool(all(abs(hsr.loc[a, "vc_dev_graph"] - hsr.loc[a, "vc_val_graph"]) <= 0.05 for a in ["shallow", "deep"]))
fin = pd.read_csv(OUT / "B_final.csv"); ep = pd.read_csv(OUT / "B_epochs.csv")
base = fin[fin.condition == "baseline"].set_index("tag"); clip = fin[fin.condition == "clipped"].set_index("tag")
b1_cells = [(t, float(clip.loc[t, "trained_hsr_balanced"]), float(base.loc[t, "recal_hsr_balanced"])) for t in clip.index]
B1 = bool(sum(abs(c - b) <= 0.02 for _, c, b in b1_cells) >= 5)
gn = ep[ep.condition == "groupnorm"]; B2 = bool((gn.eval_a2b < 0.10).all() and (gn.eval_family_f1 > 0.50).all())
C = pd.read_csv(OUT / "C_runs.csv"); ref = pd.read_csv(R / "37_audit_followups/twenty_seed_runs.csv"); ref = ref[ref.architecture == "deep"]
c_rows = []
for m in METHODS:
    a = C[C.allocation_of == m].recal_hsr_balanced; b = ref[ref.method == m].recal_hsr_balanced
    c_rows.append({"allocation_of": m, "method_hsr_mean": float(b.mean()), "allocation_random_hsr_mean": float(a.mean()), "diff": float(a.mean() - b.mean()),
                   "welch_p": float(stats.ttest_ind(a, b, equal_var=False).pvalue), "method_awbir_mean": float(ref[ref.method == m].recal_awbir.mean()), "allocation_random_awbir_mean": float(C[C.allocation_of == m].recal_awbir.mean())})
Ct = pd.DataFrame(c_rows); Ct.to_csv(OUT / "C_allocation_vs_method.csv", index=False)
C1 = bool((Ct["diff"].abs() <= 0.006).sum() >= 4)
rank_rho_alloc = float(spearmanr(Ct.method_hsr_mean, Ct.allocation_random_hsr_mean).statistic)
verdict = {"arm": "A_dev_graph__B_prevention_endpoints__C_allocation",
           "A1_vc_validity_independent_of_validation_graph": A1, "A_hsr_harm_rows": hsr[["vc_val_graph", "vc_dev_graph", "fisher", "taylor", "magnitude"]].round(4).to_dict("index"),
           "A_score_agreement": json.loads(A[A.harm == "score_agreement_vc_dev_vs_vc_val"][["architecture", "vc_dev_graph"]].round(4).to_json(orient="records")),
           "B1_clipping_hsr_within_0.02_of_recalibrated_baseline_in_5_of_6": B1, "B1_cells": [{"tag": t, "clipped_final_hsr": c, "baseline_recalibrated_hsr": b} for t, c, b in b1_cells],
           "B_final": json.loads(fin.round(4).to_json(orient="records")), "B2_groupnorm_a2b_below_0.10_and_family_f1_above_0.50_every_unit": B2,
           "B_groupnorm_ranges": {"a2b": [float(gn.eval_a2b.min()), float(gn.eval_a2b.max())], "family_f1": [float(gn.eval_family_f1.min()), float(gn.eval_family_f1.max())], "hsr": [float(gn.eval_hsr_balanced.min()), float(gn.eval_hsr_balanced.max())]},
           "B_clipped_vs_baseline_collapses": {"baseline": int(fin[fin.condition == "baseline"].collapses.sum()), "clipped": int(fin[fin.condition == "clipped"].collapses.sum())},
           "C1_allocation_reproduces_method_hsr_in_4_of_5": C1, "C_table": json.loads(Ct.round(4).to_json(orient="records")), "C_rank_rho_allocation_vs_method_means": rank_rho_alloc,
           "prereg": json.load(open(OUT / "P39_PREREGISTRATION.json"))}
(OUT / "P39_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "B_final", "A_hsr_harm_rows", "C_table")}, indent=2))
print(Ct.round(4).to_string(index=False)); print(hsr.round(3).to_string())


In [ ]:
# Stage 8 - figures
fin = pd.read_csv(OUT / "B_final.csv"); Ct = pd.read_csv(OUT / "C_allocation_vs_method.csv"); A = pd.read_csv(OUT / "A_validity_dev_graph.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
ax = axes[0]; hh = A[A.harm.isin(["harm_hsr_balanced_soc", "harm_family_macro_f1", "harm_fine_macro_f1"])]
x = np.arange(len(hh)); ax.bar(x - 0.2, hh.vc_val_graph, 0.4, label="V-C, validation graph"); ax.bar(x + 0.2, hh.vc_dev_graph, 0.4, label="V-C, development graph")
ax.set_xticks(x); ax.set_xticklabels([f"{a[:2]}/{h.replace('harm_', '').replace('_balanced_soc', '')}" for a, h in zip(hh.architecture, hh.harm)], fontsize=7, rotation=30); ax.set_ylabel("Spearman with single-channel harm"); ax.legend(fontsize=7); ax.set_title("A: IoMT validity, graph from training slice", fontsize=9)
ax = axes[1]; tags = fin[fin.condition == "baseline"].tag.tolist(); xx = np.arange(len(tags))
for off, cond, lab in [(-0.25, "baseline", "unclipped, recalibrated"), (0.0, "clipped", "clipped, as trained"), (0.25, "clipped", "clipped, recalibrated")]:
    col = "recal_hsr_balanced" if "recalibrated" in lab else "trained_hsr_balanced"; vals = [float(fin[(fin.condition == cond) & (fin.tag == t)][col].iloc[0]) for t in tags]
    ax.bar(xx + off, vals, 0.25, label=lab)
ax.set_xticks(xx); ax.set_xticklabels([t.replace("pruned/", "").replace("dense/", "dense ") for t in tags], fontsize=6, rotation=30); ax.set_ylabel("ground-truth HSR, final unit"); ax.legend(fontsize=6); ax.set_title("B: prevention versus cure, deployment endpoint", fontsize=9)
ax = axes[2]; ax.scatter(Ct.method_hsr_mean, Ct.allocation_random_hsr_mean)
for r in Ct.itertuples(): ax.annotate(r.allocation_of, (r.method_hsr_mean, r.allocation_random_hsr_mean), fontsize=7)
lim = [min(Ct.method_hsr_mean.min(), Ct.allocation_random_hsr_mean.min()) - 0.005, max(Ct.method_hsr_mean.max(), Ct.allocation_random_hsr_mean.max()) + 0.005]; ax.plot(lim, lim, ls=":", color="0.5")
ax.set_xlabel("method's own HSR (20 seeds)"); ax.set_ylabel("allocation-matched random HSR (20 seeds)"); ax.set_title("C: allocation versus channel identity, deep", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "P39_summary.png", dpi=200); plt.show(); print("written ->", OUT)


In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cat ../.gitconfig > /root/.gitconfig
cat ../.git-credentials > /root/.git-credentials && chmod 600 /root/.git-credentials
python3 - <<'EOF'
import json
nb = json.load(open("notebooks/39_endpoint_prevention_allocation.ipynb"))
for c in nb["cells"]:
    if c.get("cell_type") == "code":
        c["outputs"] = []; c["execution_count"] = None
json.dump(nb, open("/tmp/stripped.ipynb", "w"), ensure_ascii=False, indent=1)
EOF
blob=$(git hash-object -w /tmp/stripped.ipynb)
git update-index --add --cacheinfo 100644,$blob,notebooks/39_endpoint_prevention_allocation.ipynb
git add results/saber/39_endpoint_prevention_allocation
git commit -m "NB39 results: development-slice graph, prevention endpoints, allocation control. A1 HELD: with the CIC-IoMT-2024 vulnerability graph rebuilt from a 10% training-only development slice (58 edges vs 57), V-C's Spearman with single-channel HSR harm is 0.539 shallow / 0.470 deep against 0.509 / 0.510 with the validation graph (within 0.05 both); score agreement between the two V-C variants 0.80 / 0.86; the AWBIR-harm correlation at depth falls 0.778 -> 0.614 because that harm still uses the validation graph. B1 HELD: clipping the six collapsing CICIoT2023 cells gives 0 collapses (baseline 9) and final ground-truth HSR within 0.02 of the unclipped-then-recalibrated student in 6 of 6 cells (max |diff| 0.015), attack-miss 0.034-0.066 vs the cure's 0.040-0.067. B2 HELD: GroupNorm dense models keep attack-miss 0.048-0.068 and family F1 0.56-0.68 at every unit; final HSR 0.186 shallow / 0.190 deep, the shallow one worse than the recalibrated BatchNorm model's 0.162. C1 FAILED (3 of 5 within 0.006, reading required 4): allocation-matched random reproduces V-C (+0.001), Taylor (+0.003) and random (-0.006) but not magnitude (+0.018: its advantage lies in which eight first-layer channels it keeps) or Fisher (+0.006); two random structures with identical widths differ by 0.006 (p=0.001), a structure lottery that bounds every single-structure comparison at depth"
git push origin saber-ids-method
git log --oneline -1